In [1]:
geography_urls = {
    "Land_Area_sq_km": "https://www.globalfirepower.com/square-land-area.php",
    "Coastline_km": "https://www.globalfirepower.com/coastline-coverage.php",
    "Border_Length_km": "https://www.globalfirepower.com/border-coverage.php",
    "Waterways_km": "https://www.globalfirepower.com/waterway-coverage.php"
}

In [2]:
import requests
import pandas as pd
import time
import re
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load: {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value_div = record.find("div", class_="valueContainer")

        if value_div:

            if long_name:
                country = long_name.get_text(strip=True)
            elif short_name:
                country = short_name.get_text(strip=True)
            else:
                continue

            value_text = value_div.get_text(" ", strip=True)

            match = re.search(r'[\d,]+', value_text)

            if match:
                value = match.group(0)
            else:
                value = None

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in geography_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Land_Area_sq_km...
Scraping Coastline_km...
Scraping Border_Length_km...
Scraping Waterways_km...


In [6]:
geography_df = dfs[0]

for df in dfs[1:]:

    geography_df = geography_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in geography_df.columns[1:]:

    geography_df[col] = (
        geography_df[col]
        .str.replace(",", "", regex=False)
    )

    geography_df[col] = pd.to_numeric(
        geography_df[col],
        errors="coerce"
    )

In [8]:
print(geography_df.head())

       Country  Land_Area_sq_km  Coastline_km  Border_Length_km  Waterways_km
0  Afghanistan           652230             0              5987          1200
1      Albania            28748           362               691            41
2      Algeria          2381740           998              6734             0
3       Angola          1246700          1600              5369          1300
4    Argentina          2780400          4989             11968         11000


In [9]:
geography_df.to_csv("geography.csv", index=False)

print("Geography dataset saved successfully!")

Geography dataset saved successfully!


In [10]:
print("Shape:", geography_df.shape)

print()

print(geography_df.info())

print()

print(geography_df.head(10))

Shape: (145, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Country           145 non-null    object
 1   Land_Area_sq_km   145 non-null    int64 
 2   Coastline_km      145 non-null    int64 
 3   Border_Length_km  145 non-null    int64 
 4   Waterways_km      145 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 5.8+ KB
None

       Country  Land_Area_sq_km  Coastline_km  Border_Length_km  Waterways_km
0  Afghanistan           652230             0              5987          1200
1      Albania            28748           362               691            41
2      Algeria          2381740           998              6734             0
3       Angola          1246700          1600              5369          1300
4    Argentina          2780400          4989             11968         11000
5      Armenia            29743  